In [1]:
import pandas as pd
import jax as jax
import jax.numpy as jnp
from jax import grad

In [2]:
insurance_df = pd.read_csv('insurance.csv')

In [3]:
insurance_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [4]:
x_data = insurance_df.drop(columns=['charges'])
y_data = insurance_df['charges'].values

In [5]:
def linear_regression(params, x):
    return jnp.dot(x, params)

def least_squares_loss(params, x, y):
    predictions = linear_regression(params, x)

    return ((predictions - y) ** 2)

In [6]:
def prepare_data(x_df):
    x_processed = x_df.copy()
    x_processed['sex'] = x_processed['sex'].map({'male': 0, 'female': 1})
    x_processed['smoker'] = x_processed['smoker'].map({'no': 0, 'yes': 1})
    x_processed['region'] = x_processed['region'].map({'northwest': 0, 'northeast': 1, 'southwest': 2, 'southeast': 3})

    return x_processed

In [7]:
key = jax.random.PRNGKey(42)
params = jax.random.normal(key, (x_data.shape[1],))
x_data = prepare_data(x_data)
x_jax = jnp.array(x_data.values)
y_jax = jnp.array(y_data)


In [8]:
print("Input features shape:", x_jax.shape)
print("Parameters shape:", params.shape)
print("Target values shape:", y_jax.shape)

Input features shape: (1338, 6)
Parameters shape: (6,)
Target values shape: (1338,)


In [9]:
res = linear_regression(params, x_data.values)
print("Initial predictions:", res)

Initial predictions: [ 8.48927   10.280722  10.077076  ... 11.505073   7.9357176  7.2126026]


In [10]:
# LSE

initial_loss = least_squares_loss(params=params,x=x_jax,y=y_jax)

In [11]:
import jax.numpy as jnp

def normal_equation_inv(X, y):
    term1 = jnp.linalg.inv(X.T @ X)
    term2 = X.T @ y
    return term1 @ term2

In [12]:
# Minimize LSE using normal equation
params = normal_equation_inv(x_jax, y_jax)
print("Parameters from normal equation:", params)
print("Predictions from normal equation:", linear_regression(params, x_jax))


Parameters from normal equation: [  199.97943  -755.0531     53.7735    238.0462  23308.8      -336.53516]
Predictions from normal equation: [27180.566   4644.002   7078.4824 ...  3816.5244  4158.801  36315.688 ]


In [15]:
print("Initial loss:", jnp.sum(initial_loss))

final_loss = least_squares_loss(params=params,x=x_jax,y=y_jax)
print("Final loss after normal equation:", jnp.sum(final_loss))

Initial loss: 4.31387e+11
Final loss after normal equation: 5.4776984e+10
